# In-class exercise: Which drug stops the tumor?

**~30 min, Colab CPU, work in pairs.**

A tumor "culture" contains three cell types: **tumor cells**, **fibroblasts**, and **T cells**. It was treated for 72 h with vehicle or one of three drugs (A, B, C), 3 replicate wells each, then each well was profiled by scRNA-seq. You have the sequencing data and nothing else.

**Goal:** rank the three drugs as anti-tumor candidates and say what each one does.

The data are simulated, so there is a hidden ground truth. You get it at the very end. The notebook gives you code up to the first composition table. After that you get **questions only** — write the code yourself (AI is allowed; log your prompts in the last cell). Answer each question in the markdown cell *before* moving on. Do not skip ahead.


In [ ]:
!pip install -q scanpy leidenalg igraph

In [ ]:
#@title Setup and data (run; read the simulator only after the reveal)
import numpy as np, pandas as pd, scanpy as sc, anndata as ad, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
sc.settings.verbosity = 0; sc.settings.set_figure_params(dpi=90, frameon=False)

def simulate(seed=0, n_genes=1500, cells_per_well=400):
    rng = np.random.default_rng(seed)
    types = ["tumor", "fibroblast", "Tcell"]
    truth = {  # (division/day, death/day) -- hidden
        "vehicle": {"tumor": (1.00, 0.10), "fibroblast": (0.30, 0.10), "Tcell": (0.20, 0.15)},
        "drugA":   {"tumor": (1.00, 1.20), "fibroblast": (0.30, 0.12), "Tcell": (0.20, 0.15)},
        "drugB":   {"tumor": (0.05, 0.10), "fibroblast": (0.30, 0.10), "Tcell": (0.20, 0.15)},
        "drugC":   {"tumor": (0.60, 0.90), "fibroblast": (0.10, 0.90), "Tcell": (0.05, 1.00)},
    }
    n0 = {"tumor": 4000, "fibroblast": 3000, "Tcell": 3000}; days = 3
    G = np.arange(n_genes); rng.shuffle(G)
    prog = {"tumor": G[0:40], "fibroblast": G[40:80], "Tcell": G[80:120], "cycle": G[120:170], "apoptosis": G[170:200]}
    base = rng.gamma(2, 0.5, n_genes)
    X, obs, abs_counts = [], [], []
    for arm, rates in truth.items():
        for rep in range(3):
            well = f"{arm}_r{rep+1}"
            N = {t: n0[t] * np.exp((rates[t][0] - rates[t][1]) * days) * rng.lognormal(0, 0.08) for t in types}
            abs_counts.append(dict(well=well, arm=arm, **{t: int(N[t]) for t in types}))
            p = np.array([N[t] for t in types]); p /= p.sum()
            for t in rng.choice(types, size=cells_per_well, p=p):
                div, death = rates[t]; mu = base.copy(); mu[prog[t]] *= 6
                cycling = rng.random() < min(0.9, div / 1.2); dying = rng.random() < min(0.8, death / 1.5)
                if cycling: mu[prog["cycle"]] *= 5
                if dying:   mu[prog["apoptosis"]] *= 5
                lib = rng.lognormal(np.log(3000), 0.3)
                X.append(rng.negative_binomial(2, 2 / (2 + mu / mu.sum() * lib)))
                obs.append(dict(well=well, arm=arm, true_type=t, true_cycling=cycling, true_dying=dying))
    a = ad.AnnData(X=np.array(X, dtype=np.float32), obs=pd.DataFrame(obs))
    a.var_names = [f"g{i}" for i in range(n_genes)]
    a.uns["programs"] = {k: [f"g{i}" for i in v] for k, v in prog.items()}
    a.uns["truth"] = pd.DataFrame({(ar, t): r for ar, d in truth.items() for t, r in d.items()},
                                  index=["division_per_day", "death_per_day"]).T
    return a, pd.DataFrame(abs_counts).set_index("well")

_hidden, _absolute_counts = simulate()
adata = _hidden.copy(); adata.obs = adata.obs[["well", "arm"]]     # <- what you work with
markers  = {k: _hidden.uns["programs"][k] for k in ["tumor", "fibroblast", "Tcell"]}   # known marker genes
programs = {k: _hidden.uns["programs"][k] for k in ["cycle", "apoptosis"]}             # known state programs
print(adata)
print("marker sets:", {k: len(v) for k, v in markers.items()}, "| state programs:", {k: len(v) for k, v in programs.items()})


## Given: annotation and composition

Three marker gene sets are provided (`markers`), as they would be in a real experiment. We cluster, score, label, and tabulate the fraction of each cell type per well.


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata); adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=500)
sc.pp.pca(adata, n_comps=20, mask_var="highly_variable")
sc.pp.neighbors(adata, n_neighbors=15); sc.tl.umap(adata); sc.tl.leiden(adata, resolution=0.3)
for k, g in markers.items(): sc.tl.score_genes(adata, g, score_name=f"{k}_score")
adata.obs["cell_type"] = adata.obs[[f"{k}_score" for k in markers]].idxmax(1).str.replace("_score", "")
sc.pl.umap(adata, color=["leiden", "cell_type", "arm"], ncols=3, legend_loc="on data", legend_fontsize=8)


In [ ]:
ct = pd.crosstab(adata.obs["well"], adata.obs["cell_type"])
props = ct.div(ct.sum(1), axis=0)
props["arm"] = adata.obs.drop_duplicates("well").set_index("well").loc[props.index, "arm"]
display(props.groupby("arm").mean().round(3))
ax = props.drop(columns="arm").plot(kind="bar", stacked=True, figsize=(10, 3.5), width=0.85)
ax.set_ylabel("fraction of cells"); ax.legend(bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()


---
# From here on: questions only

## Step 1 — Commit

Rank drugs A, B, C by tumor fraction. Which one would you advance to the next experiment? **Write it down now.** You will be asked to compare against this later.

*Answer:*


## Step 2 — Attack the measurement

Under drug A the tumor fraction fell from 0.88 to 0.19. List every process you can think of that would produce that number **without a single tumor cell dying**.

*Answer:*


## Step 3 — Check the labels

Could tumor cells be hiding under another label in some arm? What would that look like in the UMAP or the marker scores? Check it: compare marker scores per cluster, per arm.

*Hint:* `sc.pl.dotplot` or `adata.obs.groupby(["arm","cell_type"])[[...score columns...]].mean()`.


In [ ]:
# your code


*What did you find, and does it rule relabeling in or out?*

*Answer:*


## Step 4 — Separate number from state

Fewer tumor cells can mean fewer are **born** or more **die**. Those leave different fingerprints in the transcriptome.

(a) Two gene programs are provided in `programs` (`cycle`, `apoptosis`). What pattern across the two would tell you a drug **arrests** tumor cells vs. **kills** them?

*Answer:*

(b) Score both programs per cell (`sc.tl.score_genes`), call a cell "cycling"/"dying" above a threshold of your choice, and tabulate the fraction of cycling and dying **tumor** cells per arm. Classify drug A and drug B.


In [ ]:
# your code


*Classification of A and B, with the numbers:*

*Answer:*


## Step 5 — Turn the same lens on the denominator

You scored tumor cells. Now do the same for fibroblasts and T cells in each arm. Is there an arm where the **healthy** cells look worse than the tumor?


In [ ]:
# your code


*Answer:*


## Step 6 — Confront the contradiction

Drug C has the **highest tumor fraction** of the three drugs *and* one of the **highest tumor-cell death rates**. Both are true. Reconcile them in **one sentence**. Name the mathematical property of fractions that makes this possible.

*Answer:*


## Step 7 — Ask what measurement would settle it

Sequencing gives you fractions, never numbers. What **single number from the wet lab** would convert your fractions into absolute cell counts per type?

*Answer:*

It turns out the lab did measure it — total live cells per well before loading. Run the cell below to get it, then compute absolute tumor, fibroblast, and T-cell numbers per arm and plot them against the starting tumor number (4,000 per well at t = 0).


In [ ]:
total_live_cells = _absolute_counts[["tumor", "fibroblast", "Tcell"]].sum(1).rename("total_live_cells")
display(total_live_cells.to_frame().T)


In [ ]:
# your code


*Which arms brought the tumor below its starting number? What happened to the healthy cells in each?*

*Answer:*


## Step 8 — Verdict

Final ranking. For **each** drug, one sentence stating what it does to tumor division, tumor death, and healthy cells.

Then: which drug has the best **selectivity** (tumor harm relative to healthy harm)? Is that the same drug that killed the **most** tumor cells? If not, which would you advance, and what would you want to optimize?

*Answer:*


## Step 9 — Reveal and reflect

Run the cell below only after Step 8 is written.


In [ ]:
print("True division / death rates per day:"); display(_hidden.uns["truth"])
print("\nTrue absolute cells per well at 72 h (mean of 3 wells):"); display(_absolute_counts.groupby("arm").mean().round(0))
print("\nYour annotation accuracy:", (adata.obs["cell_type"].values == _hidden.obs["true_type"].values).mean().round(3))


Compare against Step 1. Which of your answers would have been wrong if you had stopped there? Which single measurement — cell-state scores or absolute counts — changed your mind most, and why?

*Answer:*

---
### Take-home
- A cell-type fraction is a **ratio**. It moves when the numerator changes *or* when the denominator changes, and it stays put when everything changes together.
- "Stopped growing" is a claim about a **rate**. A single snapshot of composition cannot give a rate. Cell-state programs tell you *how* numbers are changing; absolute counts tell you *how much*.
- With real data there is no reveal cell. Build Steps 2–7 into your analysis before you believe a composition plot.

### AI prompt log
Paste the prompts you used, in order, one line each on what you changed and why.
```
1.
2.
```
